# CSP(1 point)
In the following code, only two of the function usages are essential. One is `is_valid(self, state)`, which checks if the current state is legal; the other is `is_satisfy(self, state)`, which is to checks if the current board meets the win condition. 
The type of state is [], which stores the tuples(a, b), representing the positions of the queens in it.

In test block, for `Solver`,  `n` indicates the size of the test while `method` indicates which bts function will be used(`bts` or `improving_bts`or`min_conflict`); for method `solve`, `render` indicates whether to use a graphical interface representation.

## Question 1: You should complete the function `bts()`. (0.8 points)
You can only use iterative way, not recursive. Using recursion will incur a **20% penalty**. You may add any function you want. 
**Remember to test the case where N=6**

## Question 2: You should complete the function `improving_bts()` or `min_conflict()`. (0.2 points)
For `improving_bts()`, you can use one or more of the following strategies: Minimum Remaining Value, Least constraining value, and Forward checking. You should have a good performance (within 2s) **when N = 16 without GUI**. 

For `min_conflict()`, you should have a good performance  (within 2s) **when N = 100 without GUI**.

### DDL: April.1st
The practice will be checked in this lab class or the next lab class(before **April.1st**) by teachers or SAs. 
#### What will be tested: 
* That you understand every line of your code, not just copy from somewhere 
* That your program compiles correctly
* Correctness of the program logic 
* That the result is obtained in a reasonable time 

#### Grading: 
* Submissions in this lab class: 1.1 points.
* Submissions on time: 1 point.


In [63]:
import numpy as np
import time

def my_range(start, end):
    if start <= end:
        return range(start, end + 1)
    else:
        return range(start, end - 1, -1)


class Problem:
    char_mapping = ('·', 'Q')

    def __init__(self, n=4):
        self.N = n

    def is_valid(self, state):
        """
        check whether the state satisfies the condition or not.
        : param state: list of points (in (row id, col id) tuple form)
        : return: bool value of valid or not
        """
        board = self.get_board(state)
        res = True
        for point in state:
            i, j = point
            condition1 = board[:, j].sum() <= 1
            condition2 = board[i, :].sum() <= 1
            condition3 = self.pos_slant_condition(board, point)
            condition4 = self.neg_slant_condition(board, point)
            res = res and condition1 and condition2 and condition3 and condition4
            if not res:
                break
        return res

    def is_satisfy(self, state):
        return self.is_valid(state) and len(state) == self.N

    def pos_slant_condition(self, board, point):
        i, j = point
        tmp = min(self.N - i - 1, j)
        start = (i + tmp, j - tmp)
        tmp = min(i, self.N - j - 1)
        end = (i - tmp,  j + tmp)
        rows = my_range(start[0], end[0])
        cols = my_range(start[1], end[1])
        return board[rows, cols].sum() <= 1

    def neg_slant_condition(self, board, point):
        i, j = point
        tmp = min(i, j)
        start = (i - tmp, j - tmp)
        tmp = min(self.N - i - 1, self.N - j - 1)
        end = (i + tmp, j + tmp)
        rows = my_range(start[0], end[0])
        cols = my_range(start[1], end[1])
        return board[rows, cols].sum() <= 1

    def get_board(self, state):
        board = np.zeros([self.N, self.N], dtype=int)
        for point in state:
            board[point] = 1
        return board

    def print_state(self, state):
        board = self.get_board(state)
        print('_' * (2 * self.N + 1))
        for row in board:
            for item in row:
                print(f'|{Problem.char_mapping[item]}', end='')
            print('|')
        print('-' * (2 * self.N + 1))

In [64]:
class Solver:
    def __init__(self, n, method):
        self.N = n
        self.problem = Problem(n)
        self.method = method

    def solve(self, render=True):
        if render:
            import pygame
            w, h = 90 * self.n + 10, 90 * self.N + 10
            screen = pygame.display.set_mode((w, h))
            screen.fill('white')
            action_generator = self.method(self.problem)
            clk = pygame.time.Clock()
            queen_img = pygame.image.load('./queen.png')
            while True:
                for event in pygame.event.get():
                    if event == pygame.QUIT:
                        exit()
                try:
                    actions = next(action_generator)
                    screen.fill('white')
                    for i in range(self.n + 1):
                        pygame.draw.rect(screen, 'black', (i * 90, 0, 10, h))
                        pygame.draw.rect(screen, 'black', (0, i * 90, w, 10))
                    for action in actions:
                        i, j = action
                        screen.blit(queen_img, (10 + 90 * j, 10 + 90 * i))
                    pygame.display.flip()
                except StopIteration:
                    pass
                clk.tick(5)
            pass
        else:
            start_time = time.time()
            for actions in self.method(self.problem):
                pass
            self.problem.print_state(actions)
            print(time.time() - start_time)

# BTS: Backtracking search

In [65]:
def bts(problem):
    action_stack = []
    tried_rows = [] 
    
    while not problem.is_satisfy(action_stack):
        col = len(action_stack)
        
        if len(tried_rows) == col:
            tried_rows.append(0)
        
        found = False
        for row in range(tried_rows[col], problem.N):
            tried_rows[col] = row + 1  
            action_stack.append((row, col))
            
            if problem.is_valid(action_stack):
                found = True
                yield action_stack
                break
            else:
                action_stack.pop()
        
        if not found:
            tried_rows.pop() 
            
            if len(action_stack) == 0:
                break  
            
            action_stack.pop()  
            yield action_stack
    
    yield action_stack

# Improving BTS 
* Which variable should be assigned next?
* In what order should its values be tried?
* Can we detect inevitable failure early?

## Minimum Remaining Value
Choose the variable with the fewest legal values in its domain
## Least constraining value
Given a variable, choose the least constraining value: the one that rules out the fewest values in the remaining variables
## Forward checking
Keep track of remaining legal values for the unassigned variables. Terminate when any variable has no legal values.

In [66]:
def improving_bts(problem):
    action_stack = []     
    tried = {}           
    available = [set(range(problem.N)) for _ in range(problem.N)]  # 每列的可用行集合
    
    def update_available():
        placed_cols = {pc for (_, pc) in action_stack} # 已经放置的列
        for c in range(problem.N):
            if c not in placed_cols:
                valid_rows = set()
                for r in range(problem.N):
                    valid = True
                    for (pr, pc) in action_stack:
                        # 检查 (r,c) 是否与已放置的皇后冲突
                        if r == pr or abs(r - pr) == abs(c - pc):
                            valid = False
                            break
                    if valid:
                        valid_rows.add(r)
                available[c] = valid_rows # 更新可用行集合
    
    def select_col():
        placed_cols = {pc for (_, pc) in action_stack}
        unassigned = [c for c in range(problem.N) if c not in placed_cols] # 选中没有放置皇后的列
        
        if not unassigned:
            return None
        
        return min(unassigned, key=lambda c: len(available[c]) if available[c] else float('inf')) # 选择可用行最少的列
    
    while not problem.is_satisfy(action_stack):
        col = select_col()
        
        if col is None:
            break
        
        if col not in tried:
            tried[col] = 0
        
        found = False
        valid_rows = sorted([r for r in available[col] if r >= tried[col]]) # sort一下，从小到大选择
        
        for row in valid_rows:
            tried[col] = row + 1
            action_stack.append((row, col))
            
            if problem.is_valid(action_stack):
                update_available()
                
                placed_cols = {pc for (_, pc) in action_stack}
                dead_end = any(len(available[c]) == 0 
                              for c in range(problem.N) 
                              if c not in placed_cols) # 此路不通
                
                if not dead_end:
                    found = True
                    yield action_stack
                    break
            
            action_stack.pop()
        
        if not found:
            if col in tried:
                del tried[col]
            
            if len(action_stack) == 0:
                break
            
            action_stack.pop()
            update_available()  
            yield action_stack # 没找到就回退一步，并更新可用行集合
    
    yield action_stack

## Local search for CSP(min-conflict algorithm)

* First generate a complete assignment for all variables (this set of assignments may conflict)

* Repeat the following steps until there are no conflicts:

    * Randomly Select a variable that causes conflicts

    * Reassign the value of this variable to another value that with the least constraint onflicts with other variables

In [67]:
def min_conflict(problem):
    import random
    
    action_stack = [(random.randint(0, problem.N - 1), c) for c in range(problem.N)]
    
    def count_conflicts(state, row, col):
        conflicts = 0
        for (r, c) in state:
            if c == col:  
                continue
            if r == row or abs(r - row) == abs(c - col):
                conflicts += 1
        return conflicts
    
    def get_conflict_variables(state):
        """返回有冲突的列的索引列表"""
        conflict_cols = set()
        for i, (r1, c1) in enumerate(state):
            for j, (r2, c2) in enumerate(state):
                if i < j:  
                    if r1 == r2 or abs(r1 - r2) == abs(c1 - c2):
                        conflict_cols.add(c1)
                        conflict_cols.add(c2)
        return list(conflict_cols)
    
    def find_best_row(col, state):
        """找到第 col 列中冲突最少的行"""
        min_conflicts = float('inf')
        best_rows = []
        
        for row in range(problem.N):
            conflicts = count_conflicts(state, row, col)
            if conflicts < min_conflicts:
                min_conflicts = conflicts
                best_rows = [row]
            elif conflicts == min_conflicts:
                best_rows.append(row)
        
        return random.choice(best_rows), min_conflicts
    
    while not problem.is_satisfy(action_stack):
        
        conflict_cols = get_conflict_variables(action_stack)
        
        if not conflict_cols:
            yield action_stack
            return
        
        col = random.choice(conflict_cols)
        
        best_row, min_conf = find_best_row(col, action_stack)
        
        for i, (r, c) in enumerate(action_stack):
            if c == col:
                action_stack[i] = (best_row, col)
                break
        
        yield action_stack
    
    yield action_stack

## test block

In [68]:
Solver(n=6, method=bts).solve(render=False)

_____________
|·|·|·|Q|·|·|
|Q|·|·|·|·|·|
|·|·|·|·|Q|·|
|·|Q|·|·|·|·|
|·|·|·|·|·|Q|
|·|·|Q|·|·|·|
-------------
0.012489557266235352


In [71]:
Solver(n=50, method=improving_bts).solve(render=False)

_____________________________________________________________________________________________________
|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·

In [72]:
Solver(n=400, method=min_conflict).solve(render=False)

_________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·